# MovieLens Analytics

### Испортируем модули и скачиваем необходимые пакеты

In [75]:
import csv
import re
import time
from collections import defaultdict
import requests
from bs4 import BeautifulSoup
import datetime

In [2]:
!pip install requests beautifulsoup4

### Создаем класс Movies, обрабатывающий первые 1000 строк из файла `movies.csv`

In [15]:

class Movies:
    def __init__(self, path_to_the_file):
        self.path_to_the_file =  path_to_the_file
        self.movies = []

        with open(path_to_the_file, "r", encoding="utf-8") as file_in:
            for line in file_in:
                line = line.strip()
                current_array = []
                first_comma = line.index(",")
                current_array.append(line[:first_comma])
                for i, item in enumerate(line):
                    if item == ",":
                        latest_comma = i
                    
                current_array.append(line[first_comma + 1 : latest_comma])
                current_array.append(line[latest_comma +1:])
                self.movies.append(current_array)

        self.movies = self.movies[:1000]
                                     
    def dist_by_release(self):
        years = []
        for line in self.movies[1:]:
            title = line[1].strip("'").strip('"')
            year = title.split()[-1]
            year = year.strip("(").strip(")")
            
            if year.isalpha():
                years.append("the year is not specified")
            else:
                years.append(year)

        release_years = {}
        for year in years:
            if year not in release_years.keys():
                release_years[year] = 1
            else:
                release_years[year] += 1

        release_years = dict(sorted(release_years.items(), key = lambda item: -item[1]))
        return release_years
    
    def dist_by_genres(self):
        array_with_genres = [line[-1].strip().split("|") for line in self.movies[1:]]
        genres = {}

        for array in array_with_genres:
            for genre in array:
                if genre not in genres.keys():
                    genres[genre] = 1
                else:
                    genres[genre] += 1

        genres = dict(sorted(genres.items(), key=lambda item : -item[1]))

        return genres
    
    def most_genres(self, n):
        movies = {}

        for movie in self.movies[1:]:
            movies[movie[1]] = len(movie[-1].split("|"))
        
        movies = dict(sorted(movies.items(), key=lambda item: -item[1])[:n])
        return movies
    
%timeit -n 1 Movies

268 ns ± 210 ns per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Вызов всех методов из класса Movies. 
#### Для метода `most_genres` можно изменить длину выводимого словаря.

In [16]:

def main_for_movies(method):
    try:
        movies = Movies("movies.csv")
        dist_by_release = movies.dist_by_release()
        dist_by_genres = movies.dist_by_genres()
        most_genres = movies.most_genres(100)

        if method == "dist_by_release":
            print(dist_by_release)
        elif method == "dist_by_genres":
            print(dist_by_genres)
        elif method == "most_genres":
            print(most_genres)
        else:
            print("This method ")
    
    except FileNotFoundError:
        print("File not found")
    except Exception as e:
        print(f"ERROR: {e}")

    

In [21]:
%timeit -n 1 -r 1 main_for_movies("dist_by_release")


{'1995': 297, '1994': 223, '1996': 180, '1993': 117, '1992': 19, '1991': 8, '1990': 7, '1939': 7, '1940': 6, '1955': 5, '1959': 5, '1968': 5, '1997': 5, '1958': 5, '1960': 4, '1943': 4, '1950': 4, '1946': 4, '1957': 4, '1954': 4, '1934': 4, '1941': 4, '1976': 3, '1988': 3, '1964': 3, '1965': 3, '1982': 3, '1937': 3, '1975': 3, '1956': 3, '1944': 3, '1953': 3, '1947': 3, '1938': 3, '1936': 3, '1967': 2, '1969': 2, '1981': 2, '1973': 2, '1987': 2, '1974': 2, '1949': 2, '1951': 2, '1961': 2, '1942': 2, '1945': 2, '1935': 2, '1977': 1, '1989': 1, '1970': 1, '1980': 1, '1986': 1, '1948': 1, '1972': 1, '1998': 1, '1933': 1, '1952': 1, '1963': 1, '1926': 1, '1932': 1, '1985': 1, '1979': 1}
134 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [22]:
%timeit -n 1 -r 1 main_for_movies("dist_by_genres")

{'Drama': 522, 'Comedy': 355, 'Romance': 198, 'Thriller': 173, 'Action': 136, 'Crime': 108, 'Adventure': 106, 'Children': 87, 'Mystery': 52, 'Sci-Fi': 52, 'Fantasy': 51, 'Horror': 43, 'Documentary': 34, 'War': 34, 'Musical': 32, 'Animation': 24, 'Western': 20, 'Film-Noir': 10, 'IMAX': 5}
140 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [23]:
%timeit -n 1 -r 1 main_for_movies("most_genres")

{'Strange Days (1995)': 6, '"Lion King, The (1994)"': 6, '"Getaway, The (1994)"': 6, 'Super Mario Bros. (1993)': 6, 'Beauty and the Beast (1991)': 6, 'All Dogs Go to Heaven 2 (1996)': 6, 'Space Jam (1996)': 6, 'Toy Story (1995)': 5, 'Money Train (1995)': 5, 'Copycat (1995)': 5, '"City of Lost Children, The (Cité des enfants perdus, La) (1995)"': 5, 'Pocahontas (1995)': 5, 'Bad Boys (1995)': 5, '"Kid in King Arthur\'s Court, A (1995)"': 5, 'True Lies (1994)': 5, 'RoboCop 3 (1993)': 5, '"Pagemaster, The (1994)"': 5, 'Ghost (1990)': 5, 'Aladdin (1992)': 5, 'Snow White and the Seven Dwarfs (1937)': 5, 'Heavy Metal (1981)': 5, 'James and the Giant Peach (1996)': 5, '"Alphaville (Alphaville, une étrange aventure de Lemmy Caution) (1965)"': 5, 'Oliver & Company (1988)': 5, '"Hunchback of Notre Dame, The (1996)"': 5, 'North by Northwest (1959)': 5, 'Charade (1963)': 5, 'Beat the Devil (1953)': 5, 'Kids of the Round Table (1995)': 4, 'From Dusk Till Dawn (1996)': 4, '"Crossing Guard, The (1995)

### Создаем класс Ratings, обрабатываюший файл `ratings.scv`

In [72]:
class Ratings:

    def __init__(self, path_to_the_file):
        self.ratings = []

        with open(path_to_the_file, "r", encoding="utf-8") as file_in:
            next(file_in)
            for line in file_in:
                self.ratings.append(line.split(","))

        
        self.Movies.ratings = self.ratings[:1000]

    class Movies:
        def __init__(self):
            movies = Movies("movies.csv")
            self.inf_from_Movies = movies.movies[:1000]
            
        def dist_by_year(self):
            years = []
            for line in self.ratings:
                dt_object = datetime.datetime.fromtimestamp(int(line[-1].strip("\n")))
                year = dt_object.year
                years.append(year) 

            ratings_by_year = {}
            for year in years:
                if year not in ratings_by_year.keys():
                    ratings_by_year[year] = 1
                else:
                    ratings_by_year[year] += 1

            ratings_by_year = dict(sorted(ratings_by_year.items(), key=lambda item: item[0]))

            return ratings_by_year
        
        def dist_by_rating(self):
            ratings_distribution = {}

            for line in self.ratings:
                mark = line[2]
                if mark not in ratings_distribution.keys():
                    ratings_distribution[mark] = 1
                else:
                    ratings_distribution[mark] += 1

            ratings_distribution = dict(sorted(ratings_distribution.items(), key=lambda item : item[0]))
            return ratings_distribution

        def top_by_num_of_ratings(self, n):
            ratings_for_movieId = {}

            for line in self.ratings:
                if line[1] not in ratings_for_movieId.keys():
                    ratings_for_movieId[line[1]] = [float(line[2])]
                else:
                    ratings_for_movieId[line[1]] += [float(line[2])]

            top_movies = {}
            
            for line in self.inf_from_Movies[1:]:
                if line[0] not in ratings_for_movieId.keys():
                    marks = []
                else:
                    marks = ratings_for_movieId[line[0]]
                top_movies[line[1]] = marks

            top_movies = dict(sorted(top_movies.items(), key=lambda item: -len(item[1]))[:n])
            return top_movies
        
        def top_by_ratings(self, n, metric="average"):
            num_of_ratings = self.top_by_num_of_ratings(len(self.inf_from_Movies)) 
            top_movies = {}
            if metric == "average":
                for movies in num_of_ratings:
                    if len(num_of_ratings[movies]) == 0:
                        top_movies[movies] = 0
                    else:
                        top_movies[movies] = round(sum(num_of_ratings[movies]) / len(num_of_ratings[movies]), 2)
            elif metric == "median":
                for movies in num_of_ratings:
                    sorted_marks = sorted(num_of_ratings[movies])
                    if len(sorted_marks) == 0: 
                        top_movies[movies] = 0.0
                    elif len(sorted_marks) == 1:
                        top_movies[movies] = sorted_marks[0]
                    else:
                        if len(sorted_marks) % 2 == 0:
                            top_movies[movies] = round((sorted_marks[len(sorted_marks) // 2 - 1] + sorted_marks[len(sorted_marks) // 2]) / 2, 2)
                        else:
                            top_movies[movies] = round(sorted_marks[len(sorted_marks) // 2], 2)
            else:
                raise ValueError("Incorrect metric")

            top_movies = dict(sorted(top_movies.items(), key=lambda item: -item[1])[:n])
            
            return top_movies
        
        def top_controversial(self, n):
            num_of_ratings = self.top_by_num_of_ratings(len(self.inf_from_Movies))
            top_movies = {}
            for movies in num_of_ratings:
                if len(num_of_ratings[movies]) < 2:
                    top_movies[movies] = 0.0
                else:
                    mean = sum(num_of_ratings[movies]) / len(num_of_ratings[movies])
                    squared_mean = [(x - mean) ** 2 for x in num_of_ratings[movies]]
                    top_movies[movies] = round(sum(squared_mean) / len(squared_mean),2)

            top_movies = dict(sorted(top_movies.items(), key=lambda item: item[1], reverse = True)[:n])
            return top_movies
        
    class User(Movies):

        def dist_user_by_marks(self):
            self.user_by_marks = {}
            for line in self.ratings[1:]:
                user = line[0]
                if user not in self.user_by_marks.keys():
                    self.user_by_marks[user] = [line[2]]
                else:
                    self.user_by_marks[user] += [line[2]]

            dist_marks_by_users = [(user, len(marks)) for user, marks in self.user_by_marks.items()]
            dist_user_by_marks = {}

            for tuple in dist_marks_by_users:
                counts = tuple[1]
                if counts not in dist_user_by_marks.keys():
                    dist_user_by_marks[counts] = 1
                else:
                    dist_user_by_marks[counts] += 1

            return dist_user_by_marks
        
        def dist_user_by_mean(self, metric="average"):
            ratings_by_user = {}
            if metric == "average":
                for user, ratings in self.user_by_marks.items():
                    if len(ratings) == 0:
                        ratings_by_user[user] = 0.0
                    elif len(ratings) == 1:
                        ratings_by_user[user] = float(ratings)
                    else:
                        ratings = [float(x) for x in ratings]
                        ratings_by_user[user] = round(sum(ratings) / len(ratings), 2)
            elif metric == "median":
                for user, ratings in self.user_by_marks.items():
                    sorted_ratings = sorted(ratings)
                    if len(sorted_ratings) == 0:
                        ratings_by_user[user] = 0.0
                    elif len(sorted_ratings) == 1:
                        ratings_by_user[user] = ratings
                    else:
                        if len(sorted_ratings) % 2 == 0:
                            ratings_by_user[user] = round((sorted_ratings[len(sorted_ratings) // 2 - 1] + sorted_ratings[len(sorted_ratings) // 2]) / 2, 2)
                        else:
                            ratings_by_user[user] = round(sorted_ratings(len(sorted_ratings) // 2), 2)
            else:
                raise ValueError("Incorrect metric")
            
            user_by_ratings = {}
            for user, digit in ratings_by_user.items():
                if digit not in user_by_ratings.keys():
                    user_by_ratings[digit] = 1
                else:
                    user_by_ratings[digit] += 1

            return user_by_ratings 
        
        def difference_in_ratings(self, n):
            difference_in_ratings = {}
            for user, ratings in self.user_by_marks.items():
                if len(ratings) < 2:
                    difference_in_ratings[user] = 0.0
                else:
                    ratings = [float(x) for x in ratings]
                    mean = sum(ratings) / len(ratings)
                    squared_mean = [(x - mean) ** 2 for x in ratings]
                    difference_in_ratings[user] = round(sum(squared_mean) / len(squared_mean), 2)

            difference_in_ratings = dict(sorted(difference_in_ratings.items(), key=lambda item: item[1], reverse = True)[:n])

            return difference_in_ratings
        
%timeit -n 1 -r 1 Ratings

291 ns ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Вызов всех методов из класса Ratings. 
#### Для метода `top_by_num_of_ratings`, `top_by_ratings`, `difference_in_ratings`  можно изменить длину выводимого словаря. В методе `top_by_num_of_ratings` так же возможно изменять параметр `metric`, он может принимать следующия значения: `average`, `median`

In [73]:
def main_for_ratings(method):
    try:
        ratings = Ratings("ratings.csv")
        if not ratings:
            raise FileNotFoundError("file not found")

        movies_in_ratings = ratings.Movies()
        dist_by_year = movies_in_ratings.dist_by_year()
        dist_by_rating = movies_in_ratings.dist_by_rating()
        top_by_num_of_ratings = movies_in_ratings.top_by_num_of_ratings(30)
        top_by_ratings = movies_in_ratings.top_by_ratings(100, metric = "average")
        top_controversial = movies_in_ratings.top_controversial(10)
        
        users = ratings.User()
        dist_user_by_marks = users.dist_user_by_marks()
        dist_user_by_mean = users.dist_user_by_mean()
        difference_in_ratings = users.difference_in_ratings(10)

        if method == "dist_by_year":
            print(dist_by_year)
        elif method == "dist_by_rating":
            print(dist_by_rating)
        elif method == "top_by_num_of_ratings":
            print(top_by_num_of_ratings)
        elif method == "top_by_ratings":
            print(top_by_ratings)
        elif method == "top_controversial":
            print(top_controversial)
        elif method == "dist_user_by_marks":
            print(dist_user_by_marks)
        elif method == "dist_user_by_mean":
            print(dist_user_by_mean)
        elif method == "difference_in_ratings":
            print(difference_in_ratings)
        else:
            print("This method doesn't exist")
    
    except FileNotFoundError as e:
        print(f"ERROR: {e}")
    except Exception as e:
        print(f"ERROR: {e}")

In [74]:
%timeit -n 1 -r 1 main_for_ratings("dist_by_year")

{2006: 254, 2015: 402, 2016: 26, 2017: 73, 2019: 245}
11.5 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [31]:
%timeit -n 1 -r 1 main_for_ratings("dist_by_rating")

{'0.5': 15, '1.0': 10, '1.5': 5, '2.0': 31, '2.5': 36, '3.0': 152, '3.5': 197, '4.0': 362, '4.5': 85, '5.0': 107}
11.7 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [32]:
%timeit -n 1 -r 1 main_for_ratings("top_by_num_of_ratings")

{'Toy Story (1995)': [3.5, 4.0, 3.0], 'Star Wars: Episode IV - A New Hope (1977)': [5.0, 4.0, 3.5], 'Pulp Fiction (1994)': [5.0, 5.0, 4.0], 'Terminator 2: Judgment Day (1991)': [4.0, 4.0, 4.0], '"Shawshank Redemption, The (1994)"': [5.0, 4.0], 'Forrest Gump (1994)': [4.5, 4.0], 'Jurassic Park (1993)': [2.0, 2.0], "Schindler's List (1993)": [5.0, 4.0], 'Blade Runner (1982)': [5.0, 4.5], 'Independence Day (a.k.a. ID4) (1996)': [3.5, 2.0], '"Godfather, The (1972)"': [3.5, 5.0], '2001: A Space Odyssey (1968)': [5.0, 4.0], '"City of Lost Children, The (Cité des enfants perdus, La) (1995)"': [4.5], 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)': [4.5], '"Usual Suspects, The (1995)"': [5.0], "Mr. Holland's Opus (1995)": [0.5], 'Braveheart (1995)': [5.0], 'Taxi Driver (1976)': [4.0], 'Apollo 13 (1995)': [4.0], 'Rob Roy (1995)': [4.5], 'Johnny Mnemonic (1995)': [4.0], 'Judge Dredd (1995)': [3.0], 'Before the Rain (Pred dozhdot) (1994)': [5.0], 'French Kiss (1995)': [4.0], 'Little Women (1994)': [0

In [33]:
%timeit -n 1 -r 1 main_for_ratings("top_by_ratings")

{'"Usual Suspects, The (1995)"': 5.0, 'Braveheart (1995)': 5.0, 'Before the Rain (Pred dozhdot) (1994)': 5.0, 'Léon: The Professional (a.k.a. The Professional) (Léon) (1994)': 5.0, 'Three Colors: Blue (Trois couleurs: Bleu) (1993)': 5.0, 'Tommy Boy (1995)': 5.0, '"Fugitive, The (1993)"': 5.0, 'Underground (1995)': 5.0, 'Ghost in the Shell (Kôkaku kidôtai) (1995)': 5.0, 'Wallace & Gromit: A Close Shave (1995)': 5.0, 'Trainspotting (1996)': 5.0, 'Blade Runner (1982)': 4.75, 'Pulp Fiction (1994)': 4.67, '"Shawshank Redemption, The (1994)"': 4.5, "Schindler's List (1993)": 4.5, '2001: A Space Odyssey (1968)': 4.5, '"City of Lost Children, The (Cité des enfants perdus, La) (1995)"': 4.5, 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)': 4.5, 'Rob Roy (1995)': 4.5, 'Clear and Present Danger (1994)': 4.5, '"Lion King, The (1994)"': 4.5, 'Shadowlands (1993)': 4.5, '"Rock, The (1996)"': 4.5, 'Casablanca (1942)': 4.5, "It's a Wonderful Life (1946)": 4.5, 'Forrest Gump (1994)': 4.25, '"Godfather, The 

In [34]:
%timeit -n 1 -r 1 main_for_ratings("top_controversial")

{'Independence Day (a.k.a. ID4) (1996)': 0.56, '"Godfather, The (1972)"': 0.56, 'Star Wars: Episode IV - A New Hope (1977)': 0.39, '"Shawshank Redemption, The (1994)"': 0.25, "Schindler's List (1993)": 0.25, '2001: A Space Odyssey (1968)': 0.25, 'Pulp Fiction (1994)': 0.22, 'Toy Story (1995)': 0.17, 'Forrest Gump (1994)': 0.06, 'Blade Runner (1982)': 0.06}
11.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [35]:
%timeit -n 1 -r 1 main_for_ratings("dist_user_by_marks")

{69: 1, 184: 1, 656: 1, 90: 1}
11.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [36]:
%timeit -n 1 -r 1 main_for_ratings("dist_user_by_mean")

{3.8: 1, 3.63: 1, 3.7: 1, 3.64: 1}
12.3 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [37]:
%timeit -n 1 -r 1 main_for_ratings("difference_in_ratings")

{'2': 2.11, '1': 0.99, '4': 0.51, '3': 0.36}
11.7 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Создаем класс Tags, обрабатываюший файл `tags.scv`

In [38]:
class Tags:
    def __init__(self, path_to_the_file):
        self.path_to_the_file = path_to_the_file
        self.tags = self._load_tags()
        
    def _load_tags(self):
        tags = []
        try:
            with open(self.path_to_the_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    tags.append({
                        'userId': int(row['userId']),
                        'movieId': int(row['movieId']),
                        'tag': row['tag'].strip(),
                        'timestamp': int(row['timestamp'])
                    })
        except Exception as e:
            print(f"Error loading tags: {e}")
        return tags
    
    def most_words(self, n):
        word_counts = {}
        for tag in self.tags:
            current_tag = tag['tag']
            word_count = len(current_tag.split())
            if current_tag not in word_counts or word_count > word_counts[current_tag]:
                word_counts[current_tag] = word_count
        return dict(sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))[:n])
    
    def longest(self, n):
        tag_lengths = {tag['tag']: len(tag['tag']) for tag in self.tags}
        return sorted(tag_lengths.keys(), key=lambda x: (-tag_lengths[x], x))[:n]
    
    def most_words_and_longest(self, n):
        most_words = set(tag for tag, _ in self.most_words(n).items())
        longest = set(self.longest(n))
        return sorted(most_words & longest)
    
    def most_popular(self, n):
        tag_counts = defaultdict(int)
        for tag in self.tags:
            tag_counts[tag['tag']] += 1
        return dict(sorted(tag_counts.items(), key=lambda x: (-x[1], x[0]))[:n])
    
    def tags_with(self, word):
        word_lower = word.lower()
        unique_tags = set()
        for tag in self.tags:
            if word_lower in tag['tag'].lower():
                unique_tags.add(tag['tag'])
        return sorted(unique_tags)
    
%timeit -n 1 -r 1 Tags

625 ns ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Вызов методов из класса Tags

In [ ]:
def main_for_tags(method):
    try:
        tags = Tags("tags.csv")
        if not tags:
            raise FileNotFoundError("file not found")

        most_words = tags.most_words(10)
        longest_tag = tags.longest(10)
        most_popular =  tags.most_popular(10)
        tags_with = tags.tags_with("action")
        
        if method == "most_words":
            print(most_words)
        elif method == "longest_tag":
            print(longest_tag)
        elif method == "most_popular":
            print(most_popular)
        elif method == "tags_with":
            print(tags_with)
        else:
            print("This method doesn't exist")
    
    except FileNotFoundError as e:
        print(f"ERROR: {e}")
    except Exception as e:
        print(f"ERROR: {e}")

In [41]:
%timeit -n 1 -r 1 main_for_tags("most_words")

{'Slutningen er også meget tynd: han får at vide, at han alligvel har arvet hele lortet, og så kører han væk i hans venners åbne sportsvogn, og så er det det. det er så kort en slutning op fattigdommen som et punktum.': 42, 'den del af filmen hvor han bare går rundt og er fattig og stort set laver ingenting går godt nok utrolig meget i tomgang og bliver meget kedelig. Hvis man skal opleve hans kedsomhed og stilleståenhed så er missionen mere end vellykket': 42, 'Nogle af de første scener hvor de to møder hinanden er noget af det mest corny jeg har set, men i kraft af fortællerforholdene i filmen er der måske en grund til dette. de er utroværdige finder man nemlig ud af': 41, 'The mill I will do to have you had a good day so I hope you have to go through this again tomorrow morning so you know what you did and what I did it to me is it just you': 41, 'der var nogle ting der sejlede lidt for den eller som ikke var helt sÃ¥ stramme som man kunne have Ã¸nsket - ting der mÃ¥ske ikke blev fu

In [42]:
%timeit -n 1 -r 1 main_for_tags("longest_tag")

['den del af filmen hvor han bare går rundt og er fattig og stort set laver ingenting går godt nok utrolig meget i tomgang og bliver meget kedelig. Hvis man skal opleve hans kedsomhed og stilleståenhed så er missionen mere end vellykket', 'den har virkelig mange sjove ting kÃ¸rende for sig. vÃ¦sentlig stÃ¦rkere end den fÃ¸rste. synes den lÃ¥ner nogle ting fra moderne sitcom og er i almindelighed meget moderne i sit indhold sÃ¥som netdating. at wifi gÃ¥r i stykker mm', 'ret interessant billede. Det er utrolig visuelt, med det smukke øje i centrum. Folks øjne og briller er et form for tematisk centrum for visualiteten som fungerer ret flot. Eller analogier, såsom lyset gennem vinduets prisme osv.', 'Den fungerer lidt som en kolage som deles ind a tekststykker mellem scenerne - selvom der er et samlende narrativ gennem alle scenerne. den har virkelig mange fine passager. den var tangerende firre en halve på visse tidspunkter', 'den bliver lidt fantastisk til sidst, når hun faktisk endelig

In [43]:
%timeit -n 1 -r 1 main_for_tags("most_popular")

{'sci-fi': 8330, 'atmospheric': 6516, 'action': 5907, 'comedy': 5702, 'surreal': 5326, 'based on a book': 5079, 'twist ending': 4820, 'funny': 4738, 'visually appealing': 4526, 'dystopia': 4257}
3.13 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [44]:
%timeit -n 1 -r 1 main_for_tags("tags_with")

['...And this film has got nothing to with the action flick starring Rutger Hauer', 'ACTION', 'About half way through it started turning into a live action cartoon.', 'Action', 'Action / Violence', 'Action Adventure', 'Action Comedy', 'Action Figures', 'Action Girl', 'Action Heroine', 'Action Movie', 'Action Scenes', 'Action Survivor', 'Action comedy', 'Action relax', 'Action thriller', 'Action, Animation, Comedy, Family, Fantasy', 'Action-filled', 'Action-packed', 'Action/Comedy', 'Actionized Sequel', 'Adventure, Action, Science Fiction', 'Amateurish Action Sequences', 'Animals - live action', 'Arresting concept & setting;their attraction/confusion/angst is understated but affecting', 'Artful action sequences', 'Best action', 'Cheap action', 'Cliche Action Scenes', 'Comic book characters live action', 'Contraction In Title', 'DANGEROUS ATTRACTION', 'Dynamic CGI Action', 'Excellent action movie', 'Extreme Action', 'Family, Action-packed', 'Family-friendly, Action-packed', 'Funniest Act

### Создаем класс Links, обрабатываюший файл `links.scv`

In [45]:
class Links:

    def __init__(self, path_to_the_file):
        self.path_to_the_file = path_to_the_file
        self.links = self._load_links()  
        self.movie_id_to_imdb_id_map = {link['movieId']: link['imdbId'] for link in self.links}
        self.movie_data_cache = {}  
        
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept-Language': 'en-US,en;q=0.9'
        })

    def _load_links(self):
        links_data = []
        max_links_to_load = 50 
        try:
            with open(self.path_to_the_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for i, row in enumerate(reader):
                    if i >= max_links_to_load: 
                        break 

                    if row.get('movieId') and row.get('imdbId'):
                        links_data.append({
                            'movieId': int(row['movieId']),
                            'imdbId': row['imdbId'].zfill(7),
                        })
        except FileNotFoundError:
            print(f"links.csv file not found at {self.path_to_the_file}")
        except Exception as e:
            print(f"Error loading links: {e}")
        
        return links_data

    def _parse_money(self, money_str):
        if not money_str:
            return 0
        money_str = re.sub(r'[^\d]', '', money_str.split('(')[0].strip())
        try:
            return int(money_str)
        except ValueError:
            return 0

    def _parse_runtime(self, runtime_str):
        if not runtime_str:
            return 0
        if isinstance(runtime_str, int):
            return runtime_str
        
        minutes = 0
        match_hours_minutes = re.search(r'(\d+)\s*h(?:our)?s?\s*(\d+)\s*min(?:ute)?s?', runtime_str, re.I)
        match_hours_only = re.search(r'(\d+)\s*h(?:our)?s?', runtime_str, re.I)
        match_minutes_only = re.search(r'(\d+)\s*min(?:ute)?s?', runtime_str, re.I)

        if match_hours_minutes:
            hours = int(match_hours_minutes.group(1))
            mins = int(match_hours_minutes.group(2))
            minutes = hours * 60 + mins
        elif match_hours_only: 
            hours = int(match_hours_only.group(1))
            minutes = hours * 60
        elif match_minutes_only: 
            mins = int(match_minutes_only.group(1))
            minutes = mins
        else: 
            try:
                cleaned_str = re.sub(r'[^\d]', '', runtime_str)
                if cleaned_str:
                    num_val = int(cleaned_str)
                    if 0 < num_val < 1000: 
                         minutes = num_val
            except ValueError:
                pass 
        return minutes

    def _scrape_imdb_data(self, imdb_id):
        if imdb_id in self.movie_data_cache:
            return self.movie_data_cache[imdb_id]

        url = f"https://www.imdb.com/title/tt{imdb_id}/"
        data = {
            'Title': None,
            'Director': [],
            'Budget': None,
            'Cumulative Worldwide Gross': None,
            'Runtime': None
        }

        print(f"Scraping: {url}")

        try:
            time.sleep(0.01)
            response = self.session.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            title_tag_h1 = soup.find('h1', attrs={'data-testid': 'hero__pageTitle'})
            if title_tag_h1:
                title_span = title_tag_h1.find('span', class_='hero__primary-text')
                data['Title'] = title_span.get_text(strip=True) if title_span else title_tag_h1.get_text(strip=True)
            if not data['Title']:
                og_title_tag = soup.find('meta', property='og:title')
                if og_title_tag and og_title_tag.get('content'):
                    data['Title'] = og_title_tag['content'].replace(' - IMDb', '').strip()

            director_li = soup.find('li', attrs={'data-testid': 'title-pc-principal-credit'})
            if director_li:
                label_span = director_li.find('span', class_='ipc-metadata-list-item__label')
                if label_span and 'Director' in label_span.get_text(strip=True):
                    director_links = director_li.select('a[href*="/name/nm"]')
                    for el in director_links:
                        name = el.get_text(strip=True)
                        if name not in data['Director']:
                            data['Director'].append(name)

            budget_li = soup.find('li', attrs={'data-testid': 'title-boxoffice-budget'})
            if budget_li:
                value_span = budget_li.find('span', class_='ipc-metadata-list-item__list-content-item')
                if value_span:
                    data['Budget'] = self._parse_money(value_span.get_text(strip=True))

            gross_li = soup.find('li', attrs={'data-testid': 'title-boxoffice-cumulativeworldwidegross'})
            if gross_li:
                value_span = gross_li.find('span', class_='ipc-metadata-list-item__list-content-item')
                if value_span:
                    data['Cumulative Worldwide Gross'] = self._parse_money(value_span.get_text(strip=True))

            runtime_li = soup.find('li', attrs={'data-testid': 'title-techspec_runtime'})
            if runtime_li:
                runtime_span = runtime_li.find('div', class_='ipc-metadata-list-item__content-container')
                if runtime_span:
                    data['Runtime'] = self._parse_runtime(runtime_span.get_text(strip=True))

            if not data['Runtime']:
                details_section = soup.find('section', attrs={'data-testid': 'Details'})
                if details_section:
                    runtime_label = details_section.find(lambda tag: tag.name == 'span' and "Runtime" in tag.get_text(strip=True) and tag.find_parent('li', attrs={'data-testid':'title-details-runtime'}))
                    if runtime_label:
                        runtime_div = runtime_label.find_parent('li').find('div', class_='ipc-metadata-list-item__content-container')
                        if runtime_div:
                            data['Runtime'] = self._parse_runtime(runtime_div.get_text(strip=True))

            self.movie_data_cache[imdb_id] = data
            return data

        except requests.exceptions.RequestException as e:
            print(f"Request error for {imdb_id}: {e}")
        except Exception as e:
            print(f"Parsing error for {imdb_id}: {e}")

        self.movie_data_cache[imdb_id] = data
        return data


    def get_imdb(self, list_of_movies, list_of_fields):
        results = []
        valid_fields = ['Title', 'Director', 'Budget', 'Cumulative Worldwide Gross', 'Runtime']

        for movie_id in list_of_movies:
            imdb_id = self.movie_id_to_imdb_id_map.get(movie_id)
            current_row = [movie_id]

            if not imdb_id:
                print(f"No IMDB ID found for movieId {movie_id}.")
                current_row.extend([None] * len(list_of_fields))
                results.append(current_row)
                continue

            scraped_data = self._scrape_imdb_data(imdb_id)
            
            for field in list_of_fields:
                if field not in valid_fields:
                    print(f"{field}' is not a supported")
                    current_row.append(None)
                    continue
                
                value = scraped_data.get(field)
                if field == 'Director' and isinstance(value, list):
                    current_row.append(", ".join(value) if value else None) 
                else:
                    current_row.append(value)
            results.append(current_row)

        results.sort(key=lambda x: x[0], reverse=True)
        return results

    def top_directors(self, n):
        director_counts = defaultdict(int)
        for i, link_entry in enumerate(self.links):
            imdb_id = link_entry['imdbId']
            data = self._scrape_imdb_data(imdb_id)
            directors = data.get('Director') 
            if directors:
                for director_name in directors:
                    if director_name:
                        director_counts[director_name.strip()] += 1
        
        sorted_directors = sorted(director_counts.items(), key=lambda item: item[1], reverse=True)
        return dict(sorted_directors[:n])

    def most_expensive(self, n):
        movie_budgets = {}
        for i, link_entry in enumerate(self.links):
            imdb_id = link_entry['imdbId']
            data = self._scrape_imdb_data(imdb_id)
            
            title = data.get('Title')
            budget = data.get('Budget')

            if title and isinstance(budget, int) and budget > 0:
                movie_budgets[title] = budget
            elif title and budget is None: 
                pass 

        sorted_budgets = sorted(movie_budgets.items(), key=lambda item: item[1], reverse=True)
        return dict(sorted_budgets[:n])

    def most_profitable(self, n):
        movie_profits = {}
        for i, link_entry in enumerate(self.links):
            imdb_id = link_entry['imdbId']
            data = self._scrape_imdb_data(imdb_id)

            title = data.get('Title')
            budget = data.get('Budget') 
            gross = data.get('Cumulative Worldwide Gross')

            if title and isinstance(budget, int) and isinstance(gross, int) and gross > 0 :
                profit = gross - budget
                movie_profits[title] = profit
        
        sorted_profits = sorted(movie_profits.items(), key=lambda item: item[1], reverse=True)
        return dict(sorted_profits[:n])

    def longest(self, n):
        movie_runtimes = {}
        for i, link_entry in enumerate(self.links):
            imdb_id = link_entry['imdbId']
            data = self._scrape_imdb_data(imdb_id)

            title = data.get('Title')
            runtime_minutes = data.get('Runtime')

            if title and isinstance(runtime_minutes, int) and runtime_minutes > 0:
                movie_runtimes[title] = runtime_minutes
        
        sorted_runtimes = sorted(movie_runtimes.items(), key=lambda item: item[1], reverse=True)
        return dict(sorted_runtimes[:n])
    
    def top_cost_per_minute(self, n):
        cost_per_minute = {}
        
        for link_entry in self.links:
            imdb_id = link_entry['imdbId']
            data = self._scrape_imdb_data(imdb_id)
            
            title = data.get('Title')
            budget = data.get('Budget', 0)
            runtime = data.get('Runtime', 1)
            
            if title and isinstance(budget, (int, float)) and budget > 0 and \
            isinstance(runtime, (int, float)) and runtime > 0:
                cpm = round(budget / runtime, 2)
                cost_per_minute[title] = cpm
                
        return dict(sorted(cost_per_minute.items(), 
                        key=lambda item: item[1], 
                        reverse=True)[:n])
    
%timeit -n 1 -r 1 Links

208 ns ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Вызов методов из класса Links

In [77]:
def main_for_links(method):
    try:
        links = Links("links.csv")
        if not links:
            raise FileNotFoundError("file not found")

        top_directors = links.top_directors(10)
        most_expensive = links.most_expensive(10)
        most_profitable = links.most_profitable(10)
        longest_link = links.longest(10)
        
        if method == "top_directors":
            print(top_directors)
        elif method == "most_expensive":
            print(most_expensive)
        elif method == "most_profitable":
            print(most_profitable)
        elif method == "longest_link":
            print(longest_link)
        else:
            print("This method doesn't exist")
    
    except FileNotFoundError as e:
        print(f"ERROR: {e}")
    except Exception as e:
        print(f"ERROR: {e}")

In [60]:
%timeit -n 1 -r 1 main_for_links("top_directors")

Scraping: https://www.imdb.com/title/tt0114709/
Scraping: https://www.imdb.com/title/tt0113497/
Scraping: https://www.imdb.com/title/tt0113228/
Scraping: https://www.imdb.com/title/tt0114885/
Scraping: https://www.imdb.com/title/tt0113041/
Scraping: https://www.imdb.com/title/tt0113277/
Scraping: https://www.imdb.com/title/tt0114319/
Scraping: https://www.imdb.com/title/tt0112302/
Scraping: https://www.imdb.com/title/tt0114576/
Scraping: https://www.imdb.com/title/tt0113189/
Scraping: https://www.imdb.com/title/tt0112346/
Scraping: https://www.imdb.com/title/tt0112896/
Scraping: https://www.imdb.com/title/tt0112453/
Scraping: https://www.imdb.com/title/tt0113987/
Scraping: https://www.imdb.com/title/tt0112760/
Scraping: https://www.imdb.com/title/tt0112641/
Scraping: https://www.imdb.com/title/tt0114388/
Scraping: https://www.imdb.com/title/tt0113101/
Scraping: https://www.imdb.com/title/tt0112281/
Scraping: https://www.imdb.com/title/tt0113845/
Scraping: https://www.imdb.com/title/tt0

In [78]:
%timeit -n 1 -r 1 main_for_links("most_expensive")

Scraping: https://www.imdb.com/title/tt0114709/
Scraping: https://www.imdb.com/title/tt0113497/
Scraping: https://www.imdb.com/title/tt0113228/
Scraping: https://www.imdb.com/title/tt0114885/
Scraping: https://www.imdb.com/title/tt0113041/
Scraping: https://www.imdb.com/title/tt0113277/
Scraping: https://www.imdb.com/title/tt0114319/
Scraping: https://www.imdb.com/title/tt0112302/
Scraping: https://www.imdb.com/title/tt0114576/
Scraping: https://www.imdb.com/title/tt0113189/
Scraping: https://www.imdb.com/title/tt0112346/
Scraping: https://www.imdb.com/title/tt0112896/
Scraping: https://www.imdb.com/title/tt0112453/
Scraping: https://www.imdb.com/title/tt0113987/
Scraping: https://www.imdb.com/title/tt0112760/
Scraping: https://www.imdb.com/title/tt0112641/
Scraping: https://www.imdb.com/title/tt0114388/
Scraping: https://www.imdb.com/title/tt0113101/
Scraping: https://www.imdb.com/title/tt0112281/
Scraping: https://www.imdb.com/title/tt0113845/
Scraping: https://www.imdb.com/title/tt0

In [62]:
%timeit -n 1 -r 1 main_for_links("most_profitable")

Scraping: https://www.imdb.com/title/tt0114709/
Scraping: https://www.imdb.com/title/tt0113497/
Scraping: https://www.imdb.com/title/tt0113228/
Scraping: https://www.imdb.com/title/tt0114885/
Scraping: https://www.imdb.com/title/tt0113041/
Scraping: https://www.imdb.com/title/tt0113277/
Scraping: https://www.imdb.com/title/tt0114319/
Scraping: https://www.imdb.com/title/tt0112302/
Scraping: https://www.imdb.com/title/tt0114576/
Scraping: https://www.imdb.com/title/tt0113189/
Scraping: https://www.imdb.com/title/tt0112346/
Scraping: https://www.imdb.com/title/tt0112896/
Scraping: https://www.imdb.com/title/tt0112453/
Scraping: https://www.imdb.com/title/tt0113987/
Scraping: https://www.imdb.com/title/tt0112760/
Scraping: https://www.imdb.com/title/tt0112641/
Scraping: https://www.imdb.com/title/tt0114388/
Scraping: https://www.imdb.com/title/tt0113101/
Scraping: https://www.imdb.com/title/tt0112281/
Scraping: https://www.imdb.com/title/tt0113845/
Scraping: https://www.imdb.com/title/tt0

In [64]:
%timeit -n 1 -r 1 main_for_links("longest_link")

Scraping: https://www.imdb.com/title/tt0114709/
Scraping: https://www.imdb.com/title/tt0113497/
Scraping: https://www.imdb.com/title/tt0113228/
Scraping: https://www.imdb.com/title/tt0114885/
Scraping: https://www.imdb.com/title/tt0113041/
Scraping: https://www.imdb.com/title/tt0113277/
Scraping: https://www.imdb.com/title/tt0114319/
Scraping: https://www.imdb.com/title/tt0112302/
Scraping: https://www.imdb.com/title/tt0114576/
Scraping: https://www.imdb.com/title/tt0113189/
Scraping: https://www.imdb.com/title/tt0112346/
Scraping: https://www.imdb.com/title/tt0112896/
Scraping: https://www.imdb.com/title/tt0112453/
Scraping: https://www.imdb.com/title/tt0113987/
Scraping: https://www.imdb.com/title/tt0112760/
Scraping: https://www.imdb.com/title/tt0112641/
Scraping: https://www.imdb.com/title/tt0114388/
Scraping: https://www.imdb.com/title/tt0113101/
Scraping: https://www.imdb.com/title/tt0112281/
Scraping: https://www.imdb.com/title/tt0113845/
Scraping: https://www.imdb.com/title/tt0